# Example 12.1 / 3: Protect One Objective First

Companion notebook for **12.1 Multiple Objectives** in the AMPL lab Documentation, section "Protect One Objective First". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Protect One Objective First

A lexicographic policy optimizes the most important measure first and then optimizes the next
one without giving up more than an accepted amount of the first. The portable pattern is
sequential optimization: minimize cost, save its optimum $z_1^*$ in `best_cost`, constrain cost
to stay within `cost_tolerance` of it, then minimize emissions. Only one objective is active in
each solve: amplpy switches with `ampl.obj[name].drop()` and `.restore()` and reads the stage-1
optimum with `.value()`.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PLANTS;                           # production plants
param unit_cost {PLANTS} >= 0;        # dollars per unit
param unit_emission {PLANTS} >= 0;    # kg CO2 per unit
param capacity {PLANTS} >= 0;         # units
param demand >= 0;                    # units
param best_cost default Infinity;     # stage-1 optimum, set by the script
param cost_tolerance >= 0;            # dollars of accepted cost increase

var Make {p in PLANTS} >= 0, <= capacity[p];  # units produced at each plant
var TotalCost = sum {p in PLANTS} unit_cost[p] * Make[p];  # production cost, dollars
var TotalEmission = sum {p in PLANTS} unit_emission[p] * Make[p];  # emissions, kg CO2

minimize CostObjective: TotalCost;    # stage-1 objective
minimize EmissionObjective: TotalEmission;  # stage-2 objective

subject to MeetDemand: sum {p in PLANTS} Make[p] = demand;  # total output covers demand exactly
# stage 2: cost within tolerance of stage 1
subject to ProtectCost: TotalCost <= best_cost + cost_tolerance;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PLANTS"] = ["A", "B"]
ampl.param["unit_cost"] = {"A": 2, "B": 4}
ampl.param["unit_emission"] = {"A": 5, "B": 1}
ampl.param["capacity"] = {"A": 8, "B": 8}
ampl.param["demand"] = 10
ampl.param["cost_tolerance"] = 4

In [5]:
# run.py
ampl.obj["EmissionObjective"].drop()
ampl.solve(solver="highs", verbose=False)
print("stage 1", ampl.solve_result, ampl.obj["CostObjective"].value())
ampl.param["best_cost"] = ampl.obj["CostObjective"].value()

ampl.obj["CostObjective"].drop()
ampl.obj["EmissionObjective"].restore()
ampl.solve(solver="highs", verbose=False)
print("stage 2", ampl.solve_result, ampl.var["Make"].to_dict())
print("cost", ampl.var["TotalCost"].value(), "emission", ampl.var["TotalEmission"].value())

stage 1 solved 24.0
stage 2 solved {'A': 6, 'B': 4}
cost 28.0 emission 34.0


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`; the `objective` command selects the active
objective:

In [6]:
%%writefile data.dat
set PLANTS := A B;
param: unit_cost unit_emission capacity :=
    A    2   5   8
    B    4   1   8 ;
param demand := 10;
param cost_tolerance := 4;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs, solver_msg 0;
objective CostObjective;
solve;
let best_cost := CostObjective;
objective EmissionObjective;
solve;
display solve_result, Make, TotalCost, TotalEmission;

solve_result = solved

Make [*] :=
A  6
B  4
;

TotalCost = 28
TotalEmission = 34

